# chunkct template: chunked CT reconstruction

Copy this notebook and fill in your own scan. Out of the box it runs on CIL's built-in simulated scan,
so you can try the whole workflow on a GPU node before touching real data.

**How chunkct works**

1. A **projection source** provides the scan geometry (CIL `AcquisitionGeometry` / `ImageGeometry`) and
   reads raw `uint16` projections for a range of angles.
2. `write_projection_chunks()` copies the projections into a compressed `projections.zarr`, split into
   `ANGLE_CHUNKS` chunks along the angle axis. Already written chunks are skipped, so it can be resumed.
3. `reconstruct()` splits the volume into blocks of `RECON_CHUNK_SHAPE`. For each block it runs CIL's FDK
   on every angle chunk and sums the contributions, then writes the block into `recon.zarr`
   (stored in chunks of `VOL_CHUNK_SHAPE`). Only one block and one angle chunk are in memory at a time.

Steps 2–3 run either here (needs a GPU) or as DTU LSF batch jobs (one GPU job per volume block).

In [ ]:
from pathlib import Path

import dask.array as da
import matplotlib.pyplot as plt
import numpy as np
import qim3d
import zarr
from cil.utilities.display import show_geometry

from chunkct import ChunkedCT, ProjectionSource
from chunkct.lsf import DTULSFChunkedCT

## 1. Define your projection source

A source needs two methods, plus one optional hook:

- `get_geometries()` → `(ag, ig)`: CIL geometries of the **full, unchunked** reconstruction.
- `read_chunk(start, end)` → `np.uint16` array of shape `(end - start, rows, cols)`: raw projections for
  angles `start` to `end - 1`.
- `process_transmission_chunk(chunk)` (optional) → `float32` array of the same shape: turn raw values into
  transmission (normalisation, flat/dark-field correction, …). It runs at reconstruction time, after the
  chunk has been converted to `float32`.

This example wraps CIL's simulated cone-beam scan (300 angles, 128 × 128 detector). For a real setup you
would read files in `read_chunk` and build the geometry from your metadata, e.g. with
`AcquisitionGeometry.create_Cone3D(source_position=..., detector_position=...).set_panel(...).set_angles(...)`.
For more examples, see the sources shipped with chunkct in `chunkct.sources` (`src/chunkct/sources/` in the
repository): `TiffProjectionSource` (reads a folder of TIFFs), `NikonZeissTiffProjectionSource` (geometry
from a Nikon `.xtekct` or ZEISS `.txrm` file) and `PickleTiffProjectionSource` (geometry from pickled CIL
objects). You can use one of them directly if it fits your data, or inherit from `TiffProjectionSource` so
`read_chunk` for a folder of TIFFs is already implemented and you only write `get_geometries`.

In [ ]:
class SimulatedSource(ProjectionSource):
    """In-memory projections from CIL's simulated cone-beam scan."""

    def __init__(self):
        from cil.utilities import dataexample

        data = dataexample.SIMULATED_CONE_BEAM_DATA.get()  # transmission values in (0, 1]
        self.ag = data.geometry
        self.ig = self.ag.get_ImageGeometry()
        # Store as uint16 "detector counts", like real raw projections
        self.projections = np.round(data.as_array() * 65535).astype(np.uint16)

    def get_geometries(self):
        return self.ag, self.ig

    def read_chunk(self, start, end):
        return self.projections[start:end]

    def process_transmission_chunk(self, chunk):
        chunk /= 65535.0  # counts -> transmission
        return chunk


source = SimulatedSource()

## 2. Inspect the source

Check the geometry and one projection before reconstructing anything.

In [ ]:
ag, ig = source.get_geometries()
print(f"projections (angles, rows, cols): {ag.shape}")
print(f"volume (z, y, x):                 {ig.shape}")
print(f"voxel size:                       {ig.voxel_size_x}")

In [ ]:
show_geometry(ag);

In [ ]:
raw = source.read_chunk(0, 1)
transmission = source.process_transmission_chunk(raw.astype(np.float32))

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, img, title in zip(axes, (raw[0], transmission[0]), ("raw", "transmission"), strict=True):
    im = ax.imshow(img, cmap="gray")
    ax.set_title(title)
    fig.colorbar(im, ax=ax)

## 3. Chunking parameters

- `ANGLE_CHUNKS`: number of chunks along the angle axis. More chunks → less projection data per FDK call.
- `RECON_CHUNK_SHAPE` `(z, y, x)`: volume block reconstructed per FDK call, limited by GPU/host memory.
  In LSF mode each block is one GPU job. Blocks at the edges may be smaller.
- `VOL_CHUNK_SHAPE` `(z, y, x)`: chunk shape of `recon.zarr` on disk, i.e. how the result is stored and
  read later (e.g. `(1, ny, nx)` for fast slice-by-slice reading). Must divide `RECON_CHUNK_SHAPE`.
- `PAD_FACTOR`: horizontal edge-padding of each projection, as a fraction of its width (e.g. `0.25`);
  reduces the bright ring at the edge of the field of view. `None` = no padding.
- `FILTER`: FDK filter: `"ram-lak"`, `"shepp-logan"`, `"cosine"`, `"hamming"` or `"hann"`.

The defaults below suit the 128³ simulated volume; for a real scan use e.g. `(500, 500, 500)` or larger.

In [ ]:
OUTPUT_DIR = Path("out/template").resolve()  # absolute, so LSF jobs find it too
ANGLE_CHUNKS = 2
RECON_CHUNK_SHAPE = (64, 64, 64)
VOL_CHUNK_SHAPE = (32, 32, 32)
PAD_FACTOR = None
FILTER = "ram-lak"

## 4a. Reconstruct here (GPU node)

In [ ]:
RUN_LOCALLY = True

if RUN_LOCALLY:
    ct = ChunkedCT(
        projection_source=source,
        output_dir=OUTPUT_DIR,
        angle_chunks=ANGLE_CHUNKS,
        recon_chunk_shape=RECON_CHUNK_SHAPE,
        vol_chunk_shape=VOL_CHUNK_SHAPE,
        pad_factor=PAD_FACTOR,
        filter=FILTER,
    )
    ct.write_projection_chunks()
    ct.reconstruct()

## 4b. Or: submit DTU LSF batch jobs

One CPU job writes the projections, then one GPU job per volume block reconstructs. Jobs use this
notebook's Python interpreter. If using `dry_run=True`, `submit_jobs` only writes the job scripts into
`OUTPUT_DIR/bjobs/run_<timestamp>/scripts` for inspection.

In [ ]:
SUBMIT_LSF = False
JOB_NAME = "MYSCAN"

if SUBMIT_LSF:
    reconstructor = DTULSFChunkedCT(
        projection_source=source,
        output_dir=OUTPUT_DIR,
        angle_chunks=ANGLE_CHUNKS,
        recon_chunk_shape=RECON_CHUNK_SHAPE,
        vol_chunk_shape=VOL_CHUNK_SHAPE,
        pad_factor=PAD_FACTOR,
        filter=FILTER,
    )
    run_dir = reconstructor.submit_jobs(job_name=JOB_NAME, dry_run=False)

In [ ]:
# Monitor submitted jobs
if SUBMIT_LSF:
    reconstructor.check_jobs(JOB_NAME)
    reconstructor.summarize_run(run_dir)

## 5. Look at the result

`recon.zarr` is opened lazily; the slicer only loads the slice being shown. The display range is taken
from the central slice, so large volumes are not scanned completely.

In [ ]:
recon = da.from_zarr(zarr.open_array(OUTPUT_DIR / "recon.zarr", mode="r"))
center_slice = recon[recon.shape[0] // 2].compute()
vmin, vmax = np.percentile(center_slice, [0.5, 99.5])
recon

In [ ]:
qim3d.viz.slicer(recon, slice_axis=0, colormap="gray", min_value=vmin, max_value=vmax)